In [2]:
import pandas as pd
import numpy as np
import glob


In [6]:
baemin_files = sorted(glob.glob("../../**/baemin*.csv", recursive=True))
coupang_files = sorted(glob.glob("../../**/coupang*.csv", recursive=True))

baemin = pd.concat([pd.read_csv(f) for f in baemin_files], ignore_index=True)
coupang = pd.concat([pd.read_csv(f) for f in coupang_files], ignore_index=True)
coupang

,order_time,total_price,advertisement_yn,ad_pct
0,2026-10-01 11:46:07,18000,Y,10
1,2026-10-01 12:06:50,21000,Y,10
2,2026-10-01 12:16:29,58000,Y,10
3,2026-10-01 12:19:49,43000,Y,10
4,2026-10-01 13:00:05,58000,N,0
...,...,...,...,...
707,2026-10-31 19:33:36,32000,N,0
708,2026-10-31 19:36:08,18000,Y,5
709,2026-10-31 19:50:22,18000,N,0
710,2026-10-31 19:59:04,34000,N,0


In [9]:
baemin = baemin.rename(columns={
    "order_at": "주문일시",
    "total_bill": "전체금액",
    "ad_yn": "광고유입여부"
})

coupang = coupang.rename(columns={
    "order_time": "주문일시",
    "total_price": "전체금액",
    "advertisement_yn": "광고유입여부"
})

baemin["플랫폼"] = "배민"
coupang["플랫폼"] = "쿠팡"
coupang["cpc"] = coupang["전체금액"] * coupang["ad_pct"] / 100
coupang["cpc"] = coupang["cpc"].astype(int)
print(baemin)
print(coupang)

                     주문일시   전체금액 광고유입여부  cpc 플랫폼
0     2026-10-01 11:48:34  26000      Y  700  배민
1     2026-10-01 11:53:22  15000      N    0  배민
2     2026-10-01 12:18:52  35000      N    0  배민
3     2026-10-01 12:28:44  35000      N    0  배민
4     2026-10-01 12:50:50  35000      N    0  배민
...                   ...    ...    ...  ...  ..
1150  2026-10-31 20:34:53  17000      N    0  배민
1151  2026-10-31 20:47:07  38000      Y  640  배민
1152  2026-10-31 21:39:35  19000      Y  690  배민
1153  2026-10-31 21:45:29  45000      N    0  배민
1154  2026-10-31 21:51:03  19000      N    0  배민

[1155 rows x 5 columns]
                    주문일시   전체금액 광고유입여부  ad_pct 플랫폼   cpc
0    2026-10-01 11:46:07  18000      Y      10  쿠팡  1800
1    2026-10-01 12:06:50  21000      Y      10  쿠팡  2100
2    2026-10-01 12:16:29  58000      Y      10  쿠팡  5800
3    2026-10-01 12:19:49  43000      Y      10  쿠팡  4300
4    2026-10-01 13:00:05  58000      N       0  쿠팡     0
..                   ...    ...    ...     ..

In [11]:
total_df = pd.concat([baemin, coupang], ignore_index=True)

total_df["주문일시"] = pd.to_datetime(total_df["주문일시"])
total_df["주문일자"] = total_df["주문일시"].dt.date

total_df = total_df.sort_values(by="주문일시").reset_index(drop=True)
total_df

,주문일시,전체금액,광고유입여부,cpc,플랫폼,ad_pct,주문일자
0,2026-10-01 11:46:07,18000,Y,1800,쿠팡,10.0,2026-10-01
1,2026-10-01 11:48:34,26000,Y,700,배민,NaN,2026-10-01
2,2026-10-01 11:53:22,15000,N,0,배민,NaN,2026-10-01
3,2026-10-01 12:06:50,21000,Y,2100,쿠팡,10.0,2026-10-01
4,2026-10-01 12:16:29,58000,Y,5800,쿠팡,10.0,2026-10-01
...,...,...,...,...,...,...,...
1862,2026-10-31 20:47:07,38000,Y,640,배민,NaN,2026-10-31
1863,2026-10-31 20:49:23,24000,N,0,쿠팡,0.0,2026-10-31
1864,2026-10-31 21:39:35,19000,Y,690,배민,NaN,2026-10-31
1865,2026-10-31 21:45:29,45000,N,0,배민,NaN,2026-10-31


In [13]:
by_date = pd.pivot_table(
    total_df,
    index="주문일자",
    columns="플랫폼",
    values="전체금액",
    aggfunc="sum",
    fill_value=0
)

by_date["합계"] = by_date["배민"] + by_date["쿠팡"]
by_date

플랫폼,배민,쿠팡,합계
주문일자,,,
2026-10-01,959000,562000,1521000
2026-10-02,1348000,904000,2252000
2026-10-03,1294000,779000,2073000
2026-10-04,1127000,928000,2055000
2026-10-05,1103000,543000,1646000
2026-10-06,1028000,576000,1604000
2026-10-07,1227000,549000,1776000
2026-10-08,946000,439000,1385000
2026-10-09,1200000,835000,2035000


In [15]:
total_df["광고여부"] = total_df["광고유입여부"] == "Y"
ratio = total_df.groupby("플랫폼")["광고여부"].mean() * 100
ad_sum = total_df.groupby("플랫폼")["cpc"].sum()

print(ratio.round(2))
print(ad_sum)

플랫폼
배민    34.98
쿠팡    51.97
Name: 광고여부, dtype: float64
플랫폼
배민    243170
쿠팡    886700
Name: cpc, dtype: int64


In [17]:
bank_file = glob.glob("../../**/bank_20261001_20261031.csv", recursive=True)[0]
bank = pd.read_csv(bank_file, encoding="cp949")

mask = (bank["거래처"] == "식자재마트") & (bank["출금액"] > 0)
food_df = bank[mask].copy()

food_df["주문일자"] = pd.to_datetime(food_df["거래일시"]).dt.date
daily = food_df.groupby("주문일자")["출금액"].sum()
daily

주문일자
2026-10-01    197100
2026-10-02    380400
2026-10-03    247600
2026-10-04    289300
2026-10-05    214400
2026-10-06    209500
2026-10-07    332800
2026-10-08    219300
2026-10-09    351300
2026-10-10    324900
2026-10-11    225000
2026-10-12    305500
2026-10-13    314100
2026-10-14    375200
2026-10-15    288500
2026-10-16    389700
2026-10-17    342000
2026-10-18    184600
2026-10-19    250100
2026-10-20    205600
2026-10-21    292400
2026-10-22    263500
2026-10-23    289100
2026-10-24    289800
2026-10-25    312300
2026-10-26    272800
2026-10-27    200400
2026-10-28    391000
2026-10-29    213300
2026-10-30    302800
2026-10-31    397300
Name: 출금액, dtype: int64

In [19]:
by_date["식자재비"] = daily
by_date["식자재비"] = by_date["식자재비"].fillna(0).astype(int)
by_date["매출-식자재비"] = by_date["합계"] - by_date["식자재비"]

result = by_date.rename_axis(columns=None).reset_index()
result

,주문일자,배민,쿠팡,합계,식자재비,매출-식자재비
0,2026-10-01,959000,562000,1521000,197100,1323900
1,2026-10-02,1348000,904000,2252000,380400,1871600
2,2026-10-03,1294000,779000,2073000,247600,1825400
3,2026-10-04,1127000,928000,2055000,289300,1765700
4,2026-10-05,1103000,543000,1646000,214400,1431600
5,2026-10-06,1028000,576000,1604000,209500,1394500
6,2026-10-07,1227000,549000,1776000,332800,1443200
7,2026-10-08,946000,439000,1385000,219300,1165700
8,2026-10-09,1200000,835000,2035000,351300,1683700
9,2026-10-10,1379000,719000,2098000,324900,1773100


In [20]:
result.to_csv("result.csv", index=False, encoding="utf-8-sig")